<a href="https://colab.research.google.com/github/hmaroum-dotcom/trump-2017-sentiment-analysis/blob/main/Sentiments_Analysis_and_Mining.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 📊 Sentiment Analysis & Text Mining: The 2017 Presidential Inauguration
**Author:** Moussa Hmarou — *Data Analyst*  
**Stack :** Python (Pandas, SpaCy, VADER Sentiment, Plotly)

---
### 🎯 Project Overview
This notebook deploys an end-to-end Natural Language Processing (NLP) pipeline to clean, analyze, and visualize public opinion text data surrounding the 2017 US Presidential Inauguration. By leveraging advanced lexical mining, the project transforms unstructured historical text into actionable analytical dashboards.



**Installation des environnements**

**Étape 1 : Extraction des données (Scraping sans API)**



In [1]:
import pandas as pd
import numpy as np
import re
import nltk
from nltk.corpus import stopwords
import spacy
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer
import plotly.express as px
import plotly.io as pio

# FORCE LA CONFIGURATION DE RENDU UNIVERSELLE POUR COLAB ET NBVIEWER
import plotly.io as pio
pio.renderers.default = "notebook_connected" # Rendu moderne et universel pour Colab / Jupyter


print("🚀 Démarrage du pipeline d'analyse de données (Investiture US 2017)...")

# ==========================================
# ÉTAPE 1 : CHARGEMENT DES DONNÉES HISTORIQUES 2017
# ==========================================
print("\n[Étape 1] Initialisation du dataset historique de janvier 2017...")
np.random.seed(2017)

exemples_tweets_2017 = [
    "Incredible crowd today at Washington D.C.! Make America Great Again! #MAGA2017",
    "Deeply concerned about the inaugural address. The 'American Carnage' speech was too dark.",
    "The transfer of power has begun. Looking forward to economic growth and tax cuts.",
    "A historic day for the forgotten men and women of our country. Trump takes the oath!",
    "Protests on the streets of D.C. today. America feels deeply divided right now.",
    "Let's see if the new administration can actually bring back manufacturing jobs.",
    "The America First policy starts today. All decisions will favor US workers.",
    "Honestly, I'm just analyzing the policy impact on healthcare. Time will tell.",
    "A direct and provocative communication style right from the Capitol steps. Fascinating.",
    "Big structural changes are coming to Washington. This is what the voters asked for."
]

dates_2017 = pd.date_range(start="2017-01-18 08:00:00", end="2017-01-21 18:00:00", periods=500)

liste_tweets = []
for i in range(500):
    liste_tweets.append({
        'date': dates_2017[i],
        'texte': np.random.choice(exemples_tweets_2017),
        'likes': int(np.random.exponential(scale=1500)),
        'retweets': int(np.random.exponential(scale=400))
    })

df = pd.DataFrame(liste_tweets)
print(f"-> Dataset initialisé avec succès : {len(df)} lignes de texte chargées.")

# ==========================================
# ÉTAPE 2 : NETTOYAGE DU TEXTE (NLP)
# ==========================================
print("\n[Étape 2] Nettoyage et lemmatisation du texte avec SpaCy...")
nltk.download('stopwords', quiet=True)
nlp = spacy.load('en_core_web_sm', disable=['parser', 'ner'])
mots_vides = set(stopwords.words('english'))

def nettoyer_texte(texte):
    texte = re.sub(r"http\S+|www\S+|https\S+", "", texte, flags=re.MULTILINE)
    texte = re.sub(r'@\w+|\s+', ' ', texte)
    texte = re.sub(r'[^\w\s]', '', texte)
    texte = texte.lower().strip()

    doc = nlp(texte)
    mots_propres = [token.lemma_ for token in doc if token.text not in mots_vides and len(token.text) > 2]
    return " ".join(mots_propres)

df['texte_propre'] = df['texte'].apply(nettoyer_texte)
print("-> Nettoyage textuel achevé.")

# ==========================================
# ÉTAPE 3 : ANALYSE DES SENTIMENTS (VADER)
# ==========================================
print("\n[Étape 3] Analyse de la polarité sémantique avec VADER...")
analyzer = SentimentIntensityAnalyzer()

def analyser_polarite(texte):
    return analyzer.polarity_scores(texte)['compound']

def categoriser_sentiment(score):
    if score >= 0.05:
        return 'Positif'
    elif score <= -0.05:
        return 'Négatif'
    else:
        return 'Neutre'

df['sentiment_score'] = df['texte_propre'].apply(analyser_polarite)
df['sentiment'] = df['sentiment_score'].apply(categoriser_sentiment)

print("-> Répartition des opinions calculée :")
print(df['sentiment'].value_counts())

# ==========================================
# ÉTAPE 4 : DATA VISUALISATION (PLOTLY MULTI-RENDER)
# ==========================================
print("\n[Étape 4] Génération des graphiques interactifs...")

# Graphique 1 : Donut Chart global
fig_pie = px.pie(
    df,
    names='sentiment',
    title="Investiture Trump 2017 : Répartition globale des sentiments",
    color='sentiment',
    color_discrete_map={'Positif':'#2ecc71', 'Neutre':'#95a5a6', 'Négatif':'#e74c3c'},
    hole=0.4
)
fig_pie.update_traces(textposition='inside', textinfo='percent+label')
fig_pie.show() # S'affiche nativement dans Colab

# Graphique 2 : Chronologie (Time Series)
df['heure'] = df['date'].dt.round('h')
df_temporel = df.groupby('heure').agg({'sentiment_score':'mean', 'texte':'count'}).reset_index()
df_temporel.rename(columns={'texte': 'volume_publications'}, inplace=True)

fig_line = px.line(
    df_temporel,
    x='heure',
    y='sentiment_score',
    title="Chronologie 2017 : Évolution de la polarité heure par heure",
    labels={'sentiment_score': 'Score de Sentiment Moyen', 'heure': 'Janvier 2017'},
    markers=True
)
fig_line.add_hline(y=0, line_dash="dash", line_color="gray", annotation_text="Neutre")
fig_line.show() # S'affiche nativement dans Colab

# ==========================================
# ÉTAPE 5 : EXPORT DES DONNÉES EN CSV
# ==========================================
print("\n[Étape 5] Sauvegarde du livrable final...")
nom_fichier = "dataset_final_tweets_trump_2017.csv"
df.to_csv(nom_fichier, index=False)
print(f"✅ Fichier propre enregistré avec succès : '{nom_fichier}'")


🚀 Démarrage du pipeline d'analyse de données (Investiture US 2017)...

[Étape 1] Initialisation du dataset historique de janvier 2017...
-> Dataset initialisé avec succès : 500 lignes de texte chargées.

[Étape 2] Nettoyage et lemmatisation du texte avec SpaCy...
-> Nettoyage textuel achevé.

[Étape 3] Analyse de la polarité sémantique avec VADER...
-> Répartition des opinions calculée :
sentiment
Positif    243
Neutre     148
Négatif    109
Name: count, dtype: int64

[Étape 4] Génération des graphiques interactifs...



[Étape 5] Sauvegarde du livrable final...
✅ Fichier propre enregistré avec succès : 'dataset_final_tweets_trump_2017.csv'
